# Goodreads Recommendation Prototype
This notebook shows how to:
1. load the Goodreads dataset,
2. inspect available book metadata and cover availability,
3. select a usable subset for import,
4. build a simple content-based recommender,
5. export the chosen subset for backend import.


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('.')
BOOKS_CSV = DATA_DIR / 'books.csv'
RATINGS_CSV = DATA_DIR / 'ratings.csv'

print('Notebook working directory:', Path.cwd())
print('Books file exists:', BOOKS_CSV.exists())
print('Ratings file exists:', RATINGS_CSV.exists())


## 1. Load Goodreads files
Read the `books.csv` and `ratings.csv` files into pandas DataFrames and inspect the first rows and column names.


In [ ]:
books = pd.read_csv(BOOKS_CSV)
ratings = pd.read_csv(RATINGS_CSV)

print('Books shape:', books.shape)
print('Ratings shape:', ratings.shape)
print('\nBooks columns:')
print(books.columns.tolist())
print('\nRatings columns:')
print(ratings.columns.tolist())

books.head()


## 2. Inspect key metadata and cover availability
Focus on the fields you want for your app: `title`, `authors`, `image_url`, `average_rating`, `ratings_count`, and `description`.


In [ ]:
summary_cols = ['book_id', 'title', 'authors', 'image_url', 'average_rating', 'ratings_count', 'description']
for col in summary_cols:
    if col not in books.columns:
        print(f'Missing column: {col}')

books[summary_cols].head(10) if all(c in books.columns for c in summary_cols) else None


## 3. Choose a usable subset
Filter for books that have a cover URL and enough ratings to be meaningful. For a first import, take the top 10,000 books by `ratings_count`.


In [ ]:
books = books[books['image_url'].notna() & (books['image_url'] != '\\N')]
books = books[books['ratings_count'].fillna(0) >= 50]
books = books.sort_values('ratings_count', ascending=False).head(10000)

print('Filtered subset size:', len(books))
books[['title', 'authors', 'average_rating', 'ratings_count', 'image_url']].head()


## 4. Normalize and map fields for backend import
Create normalized fields that match your backend model and save a prepared CSV for import.


In [ ]:
def normalize_authors(authors):
    if pd.isna(authors):
        return ''
    return authors.split('/')[0].strip()

books['author'] = books['authors'].apply(normalize_authors)
books['goodreadsId'] = books['book_id'].astype(str)
books['coverUrl'] = books['image_url']
books['reviewCount'] = books['text_reviews_count'].fillna(0).astype(int)
books['externalUrl'] = books['uri'] if 'uri' in books.columns else ''

backend_columns = [
    'goodreadsId',
    'title',
    'author',
    'description',
    'coverUrl',
    'average_rating',
    'ratings_count',
    'reviewCount',
    'externalUrl'
]

import_subset = books[backend_columns].rename(columns={
    'ratings_count': 'ratingsCount',
    'average_rating': 'averageRating'
})

import_subset.head()


## 5. Export the prepared subset
Save the selected books to a CSV that can be used by a backend import script.


In [ ]:
OUTPUT_FILE = Path('goodreads_subset_for_import.csv')
import_subset.to_csv(OUTPUT_FILE, index=False)
print('Exported subset to', OUTPUT_FILE)


## 6. Build a simple content-based recommender
Create a lightweight recommender using author + genre text similarity. This is a good starting point when you have book metadata but not strong user history.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

books['genres_text'] = books['genres'].fillna('') if 'genres' in books.columns else ''
books['combined_text'] = books['author'].fillna('') + ' ' + books['genres_text']

vectorizer = TfidfVectorizer(stop_words='english')
features = vectorizer.fit_transform(books['combined_text'])

indices = pd.Series(books.index, index=books['title']).drop_duplicates()

def recommend_books(title, top_n=5):
    if title not in indices:
        return pd.DataFrame()
    idx = indices[title]
    cosine_similarities = linear_kernel(features[idx:idx+1], features).flatten()
    sim_indices = cosine_similarities.argsort()[-top_n-1:-1][::-1]
    return books.iloc[sim_indices][['title', 'author', 'averageRating', 'ratingsCount', 'coverUrl']]

recommend_books(books['title'].iloc[0], top_n=5)


## 7. Next step: backend import script
Once the subset is exported, the next step is to build a backend loader that reads `goodreads_subset_for_import.csv` and inserts rows into your Spring Boot database.

This notebook shows how to prepare and validate the dataset before import.
